# Notebook 0 — One-time setup of a clean Python environment

**Run this notebook once, before Notebooks 1 and 2.**

Your main Anaconda installation has a broken SciPy library. Instead of repairing it, this notebook builds a **separate, clean environment** called `iomt` with the exact libraries the project needs. Nothing else on your computer is changed.

What it does:
1. Creates the environment `iomt` (Python 3.11)
2. Installs numpy, pandas, scipy, scikit-learn, matplotlib, openpyxl and pymoo into it
3. Registers it in Jupyter as **"Python (iomt)"**
4. Tests that everything imports correctly

**How to run:** choose *Run All* and wait. The first run takes about **5–15 minutes** because it downloads the libraries. Messages scroll by as it works. This is normal.

When it finishes, see the instructions at the bottom.

## Step 1 — Find conda (or fall back to a standard Python virtual environment)

In [1]:
import os, sys, json, shutil, subprocess

ENV_NAME = "iomt"
PACKAGES = ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "openpyxl", "ipykernel"]

def run(cmd):
    # Runs a command and shows its output live. Stops with a clear message if it fails.
    print("\n>>>", " ".join(cmd), "\n")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"This command failed (code {p.returncode}). Send a screenshot of the output above.")

def find_conda():
    candidates = [os.environ.get("CONDA_EXE"), shutil.which("conda")]
    base = sys.base_prefix
    candidates += [os.path.join(base, "Scripts", "conda.exe"), os.path.join(base, "condabin", "conda.bat"),
                   os.path.join(base, "bin", "conda")]
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return None

CONDA = find_conda()
print("conda found at:" if CONDA else "conda not found - a standard Python virtual environment will be used instead.", CONDA or "")

conda found at: C:\Users\Home\anaconda3\Library\bin\conda.BAT


## Step 2 — Create the environment and install the libraries

In [2]:
def conda_env_path(name):
    out = subprocess.run([CONDA, "env", "list", "--json"], capture_output=True, text=True).stdout
    for p in json.loads(out)["envs"]:
        if os.path.basename(p) == name:
            return p
    return None

if CONDA:
    env_path = conda_env_path(ENV_NAME)
    if env_path:
        print(f"Environment '{ENV_NAME}' already exists at {env_path} - it will be updated.")
    else:
        run([CONDA, "create", "-n", ENV_NAME, "python=3.11", "-y"])
        env_path = conda_env_path(ENV_NAME)
    run([CONDA, "install", "-n", ENV_NAME, "-c", "conda-forge", "-y"] + PACKAGES)
else:
    env_path = os.path.abspath(ENV_NAME + "_env")
    if not os.path.exists(env_path):
        run([sys.executable, "-m", "venv", env_path])

ENV_PY = os.path.join(env_path, "python.exe") if os.name == "nt" else os.path.join(env_path, "bin", "python")
if not os.path.exists(ENV_PY):
    ENV_PY = os.path.join(env_path, "Scripts", "python.exe")
print("\nEnvironment Python:", ENV_PY)

if not CONDA:
    run([ENV_PY, "-m", "pip", "install", "--upgrade", "pip"])
    run([ENV_PY, "-m", "pip", "install"] + PACKAGES)
run([ENV_PY, "-m", "pip", "install", "pymoo"])

Environment 'iomt' already exists at C:\Users\Home\anaconda3\envs\iomt - it will be updated.

>>> C:\Users\Home\anaconda3\Library\bin\conda.BAT install -n iomt -c conda-forge -y numpy pandas scipy scikit-learn matplotlib openpyxl ipykernel 

Jupyter detected...
3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done
WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html


# All requested packages already installed.


Environment Python: C:\Users\Home\anaconda3\envs\iomt\python.exe

>>> C:\Users\Home\anaconda3\envs\iomt\python.exe -m pip install pymoo 



## Step 3 — Register the environment in Jupyter

In [3]:
run([ENV_PY, "-m", "ipykernel", "install", "--user", "--name", ENV_NAME, "--display-name", f"Python ({ENV_NAME})"])


>>> C:\Users\Home\anaconda3\envs\iomt\python.exe -m ipykernel install --user --name iomt --display-name Python (iomt) 

Installed kernelspec iomt in C:\Users\Home\AppData\Roaming\jupyter\kernels\iomt


## Step 4 — Test that every library loads

In [4]:
test = ("import numpy, pandas, scipy, sklearn, matplotlib, openpyxl, pymoo; "
        "from scipy import stats; from scipy.sparse.linalg import svds; "
        "from sklearn.ensemble import RandomForestClassifier; "
        "from pymoo.algorithms.moo.nsga2 import NSGA2; "
        "print('numpy', numpy.__version__, '| pandas', pandas.__version__, '| scipy', scipy.__version__, "
        "'| scikit-learn', sklearn.__version__, '| pymoo', pymoo.__version__)")
run([ENV_PY, "-c", test])
print("\n" + "=" * 60)
print("SETUP COMPLETE - the environment 'Python (iomt)' is ready.")
print("=" * 60)


>>> C:\Users\Home\anaconda3\envs\iomt\python.exe -c import numpy, pandas, scipy, sklearn, matplotlib, openpyxl, pymoo; from scipy import stats; from scipy.sparse.linalg import svds; from sklearn.ensemble import RandomForestClassifier; from pymoo.algorithms.moo.nsga2 import NSGA2; print('numpy', numpy.__version__, '| pandas', pandas.__version__, '| scipy', scipy.__version__, '| scikit-learn', sklearn.__version__, '| pymoo', pymoo.__version__) 

numpy 2.4.6 | pandas 3.0.6 | scipy 1.17.1 | scikit-learn 1.9.1 | pymoo 0.6.2

SETUP COMPLETE - the environment 'Python (iomt)' is ready.


## Next steps

1. **Close this notebook.**
2. Open **1_Data_Cleaning.ipynb**. It is set to use **"Python (iomt)"** automatically.
   - Check the top-right corner of the notebook: it should say **Python (iomt)**.
   - If it says something else, choose **Kernel → Change kernel → Python (iomt)**.
3. Choose **Run All**.
4. Then open **2_Feature_Selection_Experiments.ipynb**, check the kernel the same way, and choose **Run All**.

You never need to run this setup notebook again, unless you move to another computer.